# Практикум 12. Машинный перевод, выравнивание слов (Word Alignment) и алгоритм Beam Search

В лаборатории 12 вы смотрели, как с каждой итерацией EM уточняется таблица переводов $t(f\mid e)$, и как beam search отбрасывает слабые гипотезы. Здесь повторим это в коде на маленьком русско-узбекском параллельном корпусе (перевод **с русского на узбекский**):

1. реализуем IBM Model 1 с нуля и проследим, как EM «заостряет» вероятности $t(f\mid e)$ от итерации к итерации;
2. найдём выравнивания Витерби и сверимся с `nltk.translate.IBMModel1`;
3. увидим, как разбиение узбекских слов на морфемы помогает выравниванию («uyda» ↔ «в доме»);
4. реализуем beam search на игрушечной модели следующего слова, сравним его с жадным поиском и разберёмся, зачем нужна нормировка по длине.

*Практикум к лаборатории 12 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. IBM Model 1: EM с нуля

Модель IBM Model 1 описывает, как из русского предложения $e = e_1\dots e_l$ получается узбекское $f = f_1\dots f_m$: каждое узбекское слово $f_j$ выбирает себе одно русское слово $e_{a_j}$ (или пустое слово NULL) и порождается с вероятностью $t(f_j \mid e_{a_j})$. Выравнивание $a$ мы не видим, поэтому учим $t$ алгоритмом EM:

- **E-шаг:** для каждого $f_j$ считаем, насколько вероятно, что оно пришло из каждого $e_i$: $P(a_j = i) = \dfrac{t(f_j\mid e_i)}{\sum_{i'} t(f_j\mid e_{i'})}$, и прибавляем эти доли к счётчикам $c(f, e)$;
- **M-шаг:** $t(f\mid e) = c(f, e) / \sum_{f'} c(f', e)$.

Начинаем с равномерной таблицы. Корпус — тот же тип пар, что в лаборатории: «дом» – «uy», «в доме» – «uyda», «моя книга» – «kitobim».

In [ ]:
corpus = [   # (русский, узбекский)
    ("дом", "uy"), ("большой дом", "katta uy"), ("новый дом", "yangi uy"),
    ("мой дом", "uyim"), ("в доме", "uyda"), ("в моём доме", "uyimda"),
    ("в большом доме", "katta uyda"), ("книга", "kitob"), ("большая книга", "katta kitob"),
    ("новая книга", "yangi kitob"), ("моя книга", "kitobim"), ("в книге", "kitobda"),
    ("школа", "maktab"), ("большая школа", "katta maktab"), ("моя школа", "maktabim"),
    ("в школе", "maktabda"), ("я читаю книгу", "men kitob oʻqiyman"), ("он читает книгу", "u kitob oʻqiydi"),
    ("я живу в доме", "men uyda yashayman"), ("он живёт в доме", "u uyda yashaydi"), ("он живёт в большом доме", "u katta uyda yashaydi"),
    ("я учусь в школе", "men maktabda oʻqiyman"),
]
corpus = [(s.split(), t.split()) for s, t in corpus]
print(len(corpus), "пар предложений")

In [ ]:
import numpy as np
import pandas as pd

class IBM1:
    def __init__(self, corpus):
        self.corpus = corpus
        self.src = ["NULL"] + sorted({w for s, _ in corpus for w in s})     # e: русские слова + пустое слово NULL
        self.tgt = sorted({w for _, t in corpus for w in t})                # f: узбекские слова
        self.si = {w: i for i, w in enumerate(self.src)}
        self.ti = {w: j for j, w in enumerate(self.tgt)}
        # t[e, f] = t(f|e); старт: все переводы равновероятны
        self.t = np.full((len(self.src), len(self.tgt)), 1 / len(self.tgt))
        self.history, self.loglik = [self.t.copy()], []

    def em_step(self):
        count = np.zeros_like(self.t)
        ll = 0.0
        for s, f_sent in self.corpus:
            e_ids = [0] + [self.si[w] for w in s]              # позиция 0 = NULL
            for f in f_sent:
                j = self.ti[f]
                p = self.t[e_ids, j]
                ll += np.log(p.sum() / len(e_ids))
                np.add.at(count[:, j], e_ids, p / p.sum())     # E-шаг: доли P(a_j = i) идут в счётчики
        self.t = count / count.sum(axis=1, keepdims=True)      # M-шаг: нормируем счётчики по каждому e
        self.history.append(self.t.copy())
        self.loglik.append(ll)

    def train(self, n_iter):
        for _ in range(n_iter):
            self.em_step()
        return self

    def prob(self, f, e, it=-1):
        return self.history[it][self.si[e], self.ti[f]]

    def align(self, s, f_sent):
        # для каждого f берём самое вероятное e (0 = NULL)
        e_words = ["NULL"] + s
        return [int(np.argmax([self.prob(f, e) for e in e_words])) for f in f_sent]

model = IBM1(corpus).train(20)
print(f"|V_e| = {len(model.src)} (с NULL), |V_f| = {len(model.tgt)}, старт: t = 1/{len(model.tgt)} = {1 / len(model.tgt):.3f}")
print("log-likelihood по итерациям:", " ".join(f"{x:.1f}" for x in model.loglik[:8]), "...", f"{model.loglik[-1]:.1f}")

watch = [("дом", "uy"), ("доме", "uyda"), ("в", "uyda"), ("в", "maktabda"), ("большой", "katta"),
         ("большой", "uy"), ("я", "men"), ("читаю", "oʻqiyman")]
iters = [0, 1, 2, 3, 5, 10, 20]
table = pd.DataFrame({f"it {k}": [model.prob(f, e, k) for e, f in watch] for k in iters},
                     index=[f"t({f} | {e})" for e, f in watch])
print(table.round(3).to_string())

Правдоподобие корпуса (log-likelihood) растёт с каждой итерацией — это гарантия EM. В таблице видно, как вероятности «заостряются»: вначале все переводы равновероятны ($1/|V_f| = 1/18 \approx 0.056$), уже после нескольких итераций правильный перевод забирает почти всю массу.

Посмотрите на предлог «в»: $t(\text{uyda}\mid\text{в}) \to 1$, а $t(\text{maktabda}\mid\text{в}) \to 0$. Модель привязала «в» к самому частому слову с `-da` («uyda» встречается 5 раз), а «maktabda» целиком объясняется словом «школе». Для модели «uyda» и «maktabda» — неделимые строки, общий аффикс `-da` она не видит.

## 2. Выравнивания Витерби и проверка через nltk

Лучшее выравнивание для каждого узбекского слова: $a_j = \arg\max_i t(f_j \mid e_i)$. Важное ограничение IBM Model 1: каждое узбекское слово связано **ровно с одним** русским словом. Поэтому «uyda» может связаться только с «доме», а предлог «в» остаётся без пары, хотя по смыслу «uyda» = «в доме» (одно слово ↔ два).

In [ ]:
def show_alignment(model, s, f_sent):
    a = model.align(s, f_sent)
    e_words = ["NULL"] + s
    links = "  ".join(f"{f}←{e_words[i]}" for f, i in zip(f_sent, a))
    lonely = [w for k, w in enumerate(s, start=1) if k not in a]
    print(f"{' '.join(s):<26} | {' '.join(f_sent):<26} {links}" + (f"   [без пары: {', '.join(lonely)}]" if lonely else ""))

for s, f_sent in corpus:
    show_alignment(model, s, f_sent)

Слева — часть таблицы $t(f\mid e)$ после обучения. Справа — апостериорные вероятности выравнивания $P(a_j = i)$ для одного предложения, рамкой отмечено выравнивание Витерби.

In [ ]:
import matplotlib.pyplot as plt

s, f_sent = corpus[20]
e_words = ["NULL"] + s
P = np.array([[model.prob(f, e) for f in f_sent] for e in e_words])
P = P / P.sum(axis=0)                     # P(a_j = i): нормируем по столбцу

show_e = ["дом", "доме", "в", "большой", "мой", "книга", "я", "живу", "он"]
show_f = ["uy", "uyda", "uyim", "katta", "kitob", "kitobda", "men", "yashayman", "u"]
T = np.array([[model.prob(f, e) for f in show_f] for e in show_e])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1.3, 1]})
ax[0].imshow(T, cmap="Blues", vmin=0, vmax=1)
ax[0].set_xticks(range(len(show_f)))
ax[0].set_xticklabels(show_f, rotation=45, ha="right")
ax[0].set_yticks(range(len(show_e)))
ax[0].set_yticklabels(show_e)
ax[0].set_title("t(f | e) после 20 итераций")
for i in range(len(show_e)):
    for j in range(len(show_f)):
        if T[i, j] >= 0.05:
            ax[0].text(j, i, f"{T[i, j]:.2f}", ha="center", va="center", fontsize=8,
                       color="white" if T[i, j] > 0.6 else "black")
ax[1].imshow(P, cmap="Oranges", vmin=0, vmax=1)
for j, i in enumerate(model.align(s, f_sent)):
    ax[1].add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=2))
for i in range(len(e_words)):
    for j in range(len(f_sent)):
        ax[1].text(j, i, f"{P[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax[1].set_xticks(range(len(f_sent)))
ax[1].set_xticklabels(f_sent)
ax[1].set_yticks(range(len(e_words)))
ax[1].set_yticklabels(e_words)
ax[1].set_title("P(a_j = i) и выравнивание Витерби")
plt.tight_layout()
plt.show()

Та же модель в `nltk`. Обратите внимание на порядок аргументов: `AlignedSent(words, mots)`, где `words` — предложение, которое порождается (у нас узбекское), а `mots` — исходное (русское). Пустое слово в nltk обозначено `None`.

In [ ]:
from nltk.translate import IBMModel1, AlignedSent

bitext = [AlignedSent(f_sent, s) for s, f_sent in corpus]    # nltk: words = порождаемое (узб.), mots = исходное (рус.)
ibm = IBMModel1(bitext, 20)

# сравниваем пары слов, которые встречаются вместе (остальные nltk не обновляет)
cooc = {(e, f) for s, f_sent in corpus for e in ["NULL"] + s for f in f_sent}
diff = max(abs(ibm.translation_table[f][None if e == "NULL" else e] - model.prob(f, e)) for e, f in cooc)
print(f"max |t_ours - t_nltk| = {diff:.2e}  (по {len(cooc)} парам)")

same = 0
for (s, f_sent), b in zip(corpus, bitext):
    ours = {(j, i - 1 if i > 0 else None) for j, i in enumerate(model.align(s, f_sent))}
    same += ours == set(b.alignment)
print(f"совпало выравниваний: {same} / {len(corpus)}")
print(bitext[-2].words, bitext[-2].mots, "->", bitext[-2].alignment)

## 3. Морфемы помогают выравниванию

В узбекском «uyda» = «uy» + «-da», и именно «-da» соответствует русскому «в». Если разрезать узбекские слова на морфемы, у модели появляется шанс связать «в» с «+da», а «мой» — с «+im». Ниже очень простой сегментатор: он отрезает `-da` и `-im`, если остаётся хотя бы 2 буквы. Он написан только для этого корпуса; готовых морфологических анализаторов для узбекского мало, и в реальной системе обычно используют подслова (BPE, тема 2).


После обучения у нас есть словарь «русское слово → узбекская морфема». Можно ли переводить пословно? Попробуем на фразах, которых **нет** в корпусе. Видны две проблемы: порядок слов (в узбекском послелоги и аффиксы стоят после слова) и склейка морфем. Настоящая система перевода должна выбирать порядок и формы, сравнивая много вариантов целого предложения. Этим занимается декодер.

In [ ]:
def segment(word, suffixes=("da", "im")):
    # отрезаем -da и -im с конца слова: uyimda -> uy +im +da
    out = []
    peeled = True
    while peeled:
        peeled = False
        for suf in suffixes:
            if word.endswith(suf) and len(word) - len(suf) >= 2:
                out.insert(0, "+" + suf)
                word = word[: -len(suf)]
                peeled = True
    return [word] + out

def segment_sentence(words):
    return [m for w in words for m in segment(w)]

seg_corpus = [(s, segment_sentence(f_sent)) for s, f_sent in corpus]
seg_model = IBM1(seg_corpus).train(20)
for k in [3, 4, 5, 6, 20]:
    show_alignment(seg_model, *seg_corpus[k])
print()
for e, f in [("в", "+da"), ("мой", "+im"), ("моя", "+im"), ("доме", "uy"), ("доме", "+da")]:
    print(f"t({f} | {e}) = {seg_model.prob(f, e):.3f}")

In [ ]:
def word_by_word(model, words):
    # для каждого русского слова берём самую вероятную узбекскую морфему
    return [model.tgt[int(np.argmax(model.t[model.si[w]]))] for w in words if w in model.si]

gold = {"в большой школе": "katta maktabda", "моя большая книга": "katta kitobim"}
for phrase in gold:
    words = phrase.split()
    print(f"{phrase:<18} → {' '.join(word_by_word(seg_model, words)):<24} (правильно: {gold[phrase]})")

## 4. Beam search: декодер

Нейросетевой переводчик порождает перевод слово за словом: на каждом шаге модель даёт распределение $P(y_t \mid y_{<t}, x)$ над следующим словом. Вероятность всего перевода — произведение: $\log P(y\mid x) = \sum_t \log P(y_t \mid y_{<t}, x)$.

- **Жадный поиск** берёт самое вероятное слово на каждом шаге — быстро, но может пропустить лучший перевод.
- **Beam search** держит $k$ лучших частичных гипотез (луч). На каждом шаге он продолжает каждую из них всеми словами, сортирует и оставляет $k$ лучших. Гипотеза, закончившаяся `</s>`, откладывается, а луч сужается на 1.

Ниже игрушечная модель для перевода «Я читаю новую книгу»: для каждого префикса задано распределение следующего узбекского слова (числа придуманы для примера).

In [ ]:
EOS = "</s>"
# Я читаю новую книгу  ->  P(следующее слово | уже написанный префикс)
lm = {
    (): {"men": .40, "kitob": .35, "yangi": .25},
    ("men",): {"kitob": .45, "yangi": .40, "oʻqiyman": .15},
    ("kitob",): {"oʻqiyman": .70, EOS: .30},
    ("yangi",): {"kitob": .90, EOS: .10},
    ("men", "kitob"): {"oʻqiyman": .70, EOS: .30},
    ("men", "yangi"): {"kitob": .95, EOS: .05},
    ("men", "oʻqiyman"): {EOS: 1.0},
    ("kitob", "oʻqiyman"): {EOS: 1.0},
    ("yangi", "kitob"): {"oʻqiyman": .70, EOS: .30},
    ("men", "kitob", "oʻqiyman"): {EOS: 1.0},
    ("men", "yangi", "kitob"): {"oʻqiyman": .80, EOS: .20},
    ("yangi", "kitob", "oʻqiyman"): {EOS: 1.0},
    ("men", "yangi", "kitob", "oʻqiyman"): {EOS: 1.0},
}

import math

def beam_search(lm, k, alpha=0.0, verbose=False):
    beams = [((), 0.0)]                 # (гипотеза, log P)
    finished = []
    step = 0
    while beams and len(finished) < k:
        step += 1
        cands = [(h + (w,), lp + math.log(p)) for h, lp in beams for w, p in lm[h].items()]
        cands.sort(key=lambda c: -c[1])
        room = k - len(finished)        # сколько мест в луче осталось
        kept, pruned = cands[:room], cands[room:]
        beams = [c for c in kept if c[0][-1] != EOS]
        finished += [c for c in kept if c[0][-1] == EOS]
        if verbose:
            print(f"шаг {step}:")
            for h, lp in kept:
                tag = "(закончена)" if h[-1] == EOS else ""
                print(f"   + {' '.join(h):<34} log P = {lp:6.2f} {tag}")
            for h, lp in pruned:
                print(f"   - {' '.join(h):<34} log P = {lp:6.2f}   отброшена")
    score = lambda c: c[1] / len(c[0]) ** alpha    # нормировка по длине; alpha = 0 — без неё
    return max(finished, key=score), finished

best, _ = beam_search(lm, k=2, verbose=True)
print("\nлучшая законченная:", " ".join(best[0]), f"  P = {math.exp(best[1]):.3f}")

Сравним жадный поиск ($k=1$), beam search с разной шириной $k$ и **точный** поиск полным перебором всех гипотез (дерево маленькое, поэтому перебор возможен; в настоящей модели гипотез астрономически много).

**Нормировка по длине.** Каждое слово умножает вероятность на число < 1, поэтому короткие гипотезы почти всегда вероятнее длинных: «kitob oʻqiyman» (без «men» и без «yangi») выигрывает по $P$. Нормировка делит $\log P$ на длину в степени $\alpha$: $\text{score} = \log P / L^{\alpha}$ ($L$ считает и `</s>`). При $\alpha = 1$ это средний логарифм вероятности на слово.

In [ ]:
def all_hypotheses(lm, h=(), lp=0.0):
    # полный перебор дерева: все законченные гипотезы
    if h and h[-1] == EOS:
        return [(h, lp)]
    return [x for w, p in lm[h].items() for x in all_hypotheses(lm, h + (w,), lp + math.log(p))]

hyps = all_hypotheses(lm)
print(f"всего гипотез: {len(hyps)}, сумма их вероятностей = {sum(math.exp(lp) for _, lp in hyps):.3f}")
for alpha in (0.0, 1.0):
    exact = max(hyps, key=lambda c: c[1] / len(c[0]) ** alpha)
    print(f"\nalpha = {alpha:g}   точный ответ перебором: {' '.join(exact[0])}")
    for k in range(1, 6):
        (h, lp), _ = beam_search(lm, k, alpha)
        print(f"   k = {k}: {' '.join(h):<32} P = {math.exp(lp):.3f}   log P / L^alpha = {lp / len(h) ** alpha:6.3f}")

Что видно: жадный поиск теряет «yangi» (новую), потому что на втором шаге после «men» слово «kitob» (0.45) чуть вероятнее, чем «yangi» (0.40). Без нормировки широкий луч находит самую вероятную, но слишком короткую гипотезу. С нормировкой и $k \ge 4$ находится полный перевод «men yangi kitob oʻqiyman». Слева на графике — вероятность гипотезы, справа — средняя вероятность на слово $\exp(\log P / L)$; оранжевым отмечен победитель.

In [ ]:
import matplotlib.pyplot as plt

hyps.sort(key=lambda c: c[1])
labels = [" ".join(h[:-1]) for h, _ in hyps]
raw = [math.exp(lp) for _, lp in hyps]
norm = [math.exp(lp / len(h)) for h, lp in hyps]       # средняя вероятность на слово

fig, ax = plt.subplots(1, 2, figsize=(11, 0.35 * len(hyps) + 1.2), sharey=True)
for a, vals, title in [(ax[0], raw, "P(гипотезы)"), (ax[1], norm, "exp(log P / L): на слово")]:
    colors = ["#D9480F" if v == max(vals) else "#2B55C9" for v in vals]
    a.barh(range(len(vals)), vals, color=colors)
    a.set_title(title)
ax[0].set_yticks(range(len(labels)))
ax[0].set_yticklabels(labels)
plt.tight_layout()
plt.show()

Качество перевода обычно измеряют метрикой BLEU — долей совпадающих n-грамм с эталонным переводом. Подробно её разберём в практикуме 18; здесь только быстрый взгляд через `nltk` (для коротких предложений нужна сглаживающая функция).

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = "men yangi kitob oʻqiyman".split()
smooth = SmoothingFunction().method1
for k, alpha in [(1, 0.0), (2, 0.0), (4, 1.0)]:
    (h, _), _ = beam_search(lm, k, alpha)
    hyp = list(h[:-1])
    print(f"k={k}, alpha={alpha:g}: {' '.join(hyp):<28} BLEU = {sentence_bleu([reference], hyp, smoothing_function=smooth):.3f}")

## 5. Как это выглядит на настоящей модели

`facebook/nllb-200-distilled-600M` — открытая многоязычная модель перевода от Meta (600 млн параметров, файл ≈2.5 ГБ, лицензия CC BY-NC 4.0 — только некоммерческое использование). Русский в ней обозначен кодом `rus_Cyrl`, узбекский (латиница) — `uzn_Latn`. Код целевого языка передаётся через `forced_bos_token_id`, ширина луча — `num_beams`, нормировка по длине — параметр `length_penalty` (по умолчанию 1.0, то есть та же $\log P / L$). Ячейка работает только в Colab (нужен интернет, загрузка занимает несколько минут). Посмотрите, отличается ли жадный перевод от перевода лучом и какие гипотезы оказались в луче.

In [ ]:
# настоящий переводчик ru -> uz; модель скачивается один раз (≈2.5 ГБ)
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    name = "facebook/nllb-200-distilled-600M"
    tok = AutoTokenizer.from_pretrained(name, src_lang="rus_Cyrl")
    nllb = AutoModelForSeq2SeqLM.from_pretrained(name)
    tgt_id = tok.convert_tokens_to_ids("uzn_Latn")

    sentences = ["Я живу в большом доме.", "Мы читаем новую книгу в библиотеке.",
                 "Самарканд — древний город на Великом шёлковом пути."]
    for s in sentences:
        inputs = tok(s, return_tensors="pt")
        greedy = nllb.generate(**inputs, forced_bos_token_id=tgt_id, num_beams=1, max_new_tokens=60)
        beam = nllb.generate(**inputs, forced_bos_token_id=tgt_id, num_beams=4, max_new_tokens=60)
        print(s)
        print("   greedy:  ", tok.decode(greedy[0], skip_special_tokens=True))
        print("   beam=4:  ", tok.decode(beam[0], skip_special_tokens=True))

    # все 4 гипотезы луча для первого предложения; число = log P / L (length_penalty = 1.0)
    inputs = tok(sentences[0], return_tensors="pt")
    out = nllb.generate(**inputs, forced_bos_token_id=tgt_id, num_beams=4, num_return_sequences=4,
                        output_scores=True, return_dict_in_generate=True, max_new_tokens=60)
    print()
    for seq, sc in zip(out.sequences, out.sequences_scores):
        print(f"   {float(sc):7.3f}  {tok.decode(seq, skip_special_tokens=True)}")
except Exception as e:
    print("Не удалось загрузить модель (нужен интернет и пакет transformers):", e)

## 6. Задания

1. Добавьте в `corpus` пары про сад: `("сад", "bogʻ")`, `("большой сад", "katta bogʻ")`, `("в саду", "bogʻda")`, `("мой сад", "bogʻim")`. Переобучите обе модели (без сегментации и с ней). Какие три перевода самые вероятные для «сад» и «саду»? Как выровнялась фраза «в саду»?
2. Обучите `IBM1` на 30 итераций и постройте график `model.loglik` по итерациям. После какой итерации правдоподобие меняется меньше чем на 0.01?
3. **Симметризация.** Обучите модель в обратном направлении (поменяйте местами предложения в парах корпуса). Для пары «он живёт в большом доме» получите множества связей (позиция русского слова, позиция узбекского) в обоих направлениях и найдите их пересечение и объединение. Какие связи есть только в объединении?
4. Замените нормировку в `beam_search` на штраф GNMT: $lp(L) = \left(\frac{5 + L}{6}\right)^{\alpha}$, $\text{score} = \log P / lp(L)$. Что найдёт beam search с $k = 4$ при $\alpha = 0, 0.5, 1, 1.5$?
5. Посчитайте, сколько гипотез (продолжений) вычисляет `beam_search` при $k = 1 \dots 5$, и сравните с числом узлов полного дерева. Во сколько раз луч дешевле перебора на этом примере?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь